# Day 7 - Insurance Claims Analysis

## Overview
This dataset contains insightful information related to insurance claims, giving us an in-depth look into the demographic patterns of those receiving them. The dataset contains information on patient age, gender, BMI (Body Mass Index), blood pressure levels, diabetic status, number of children, smoking status and region.

By analyzing these key factors across geographical areas and across different demographics such as age or gender we can gain a greater understanding of who is most likely to receive an insurance claim.

In this notebook, we will:
- Analyze insurance claims data
- Explore relationships between features
- Build and compare multiple regression models
- Evaluate model performance

**Author:** [BELYAGOUBIABDELILAH](https://github.com/BELYAGOUBIABDELILAH)

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load the data
data = pd.read_csv('data/insurance_data.csv')
data.head()

In [ ]:
# Data info
data.info()

In [ ]:
# Statistical view
data.describe()

In [ ]:
# Check for null values
print('Null values:')
print(data.isnull().sum())

In [ ]:
# Handle missing values
data = data.interpolate()  # numeric values
data = data.fillna(data.mode().iloc[0])  # categorical features
print('After handling nulls:')
print(data.isnull().sum())

In [ ]:
# Remove unnecessary columns
if 'index' in data.columns:
    data.drop(columns=['index'], inplace=True)
if 'PatientID' in data.columns:
    data.drop(columns=['PatientID'], inplace=True)
data.head()

## Exploratory Data Analysis

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 10))
sns.heatmap(data=data.corr(), annot=True, cmap='crest')
plt.title('Feature Correlation Matrix')
plt.show()

In [ ]:
# Pairplot for key features
cols = ['age', 'bmi', 'bloodpressure', 'children', 'claim']
available_cols = [col for col in cols if col in data.columns]
if available_cols:
    sns.pairplot(data[available_cols], height=2.5)
    plt.show()

## Model Building & Evaluation

In [ ]:
# Prepare data for modeling
new_data = data.copy()

# Label encoding for categorical features
le = LabelEncoder()
categorical_cols = ['gender', 'diabetic', 'smoker', 'region']
for col in categorical_cols:
    if col in new_data.columns:
        new_data[col] = le.fit_transform(new_data[col])

new_data.head()

In [ ]:
# Feature scaling
scaler = StandardScaler()
scaler_data = scaler.fit_transform(new_data.drop(columns=['claim']))
print('Scaled data shape:', scaler_data.shape)

In [ ]:
# Train-test split
x_train, x_test, y_train, y_test = train_test_split(
    scaler_data, new_data['claim'], test_size=0.2, random_state=42
)
print(f'Train shape: {x_train.shape}, Test shape: {x_test.shape}')

In [ ]:
# Evaluation functions
def rmse_cv(model):
    rmse = np.sqrt(-cross_val_score(
        model, scaler_data, new_data['claim'], 
        scoring='neg_mean_squared_error', cv=5
    )).mean()
    return rmse

def evaluation(y, predictions):
    mae = mean_absolute_error(y, predictions)
    mse = mean_squared_error(y, predictions)
    rmse = np.sqrt(mse)
    r_squared = r2_score(y, predictions)
    return mae, mse, rmse, r_squared

# Store results
models = pd.DataFrame(columns=['Model', 'MAE', 'MSE', 'RMSE', 'R2 Score', 'RMSE (CV)'])

In [ ]:
# Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(x_train, y_train)
predictions = lin_reg.predict(x_test)

mae, mse, rmse, r2 = evaluation(y_test, predictions)
rmse_cv_score = rmse_cv(lin_reg)

models = pd.concat([models, pd.DataFrame([{
    'Model': 'LinearRegression', 'MAE': mae, 'MSE': mse, 
    'RMSE': rmse, 'R2 Score': r2, 'RMSE (CV)': rmse_cv_score
}])], ignore_index=True)

print(f'Linear Regression - RMSE: {rmse:.2f}, R2: {r2:.4f}')

In [ ]:
# Ridge Regression
ridge = Ridge()
ridge.fit(x_train, y_train)
predictions = ridge.predict(x_test)

mae, mse, rmse, r2 = evaluation(y_test, predictions)
rmse_cv_score = rmse_cv(ridge)

models = pd.concat([models, pd.DataFrame([{
    'Model': 'Ridge', 'MAE': mae, 'MSE': mse,
    'RMSE': rmse, 'R2 Score': r2, 'RMSE (CV)': rmse_cv_score
}])], ignore_index=True)

print(f'Ridge - RMSE: {rmse:.2f}, R2: {r2:.4f}')

In [ ]:
# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(x_train, y_train)
predictions = rf.predict(x_test)

mae, mse, rmse, r2 = evaluation(y_test, predictions)
rmse_cv_score = rmse_cv(rf)

models = pd.concat([models, pd.DataFrame([{
    'Model': 'RandomForest', 'MAE': mae, 'MSE': mse,
    'RMSE': rmse, 'R2 Score': r2, 'RMSE (CV)': rmse_cv_score
}])], ignore_index=True)

print(f'Random Forest - RMSE: {rmse:.2f}, R2: {r2:.4f}')

In [ ]:
# XGBoost
xgb = XGBRegressor(n_estimators=1000, learning_rate=0.01, random_state=42)
xgb.fit(x_train, y_train)
predictions = xgb.predict(x_test)

mae, mse, rmse, r2 = evaluation(y_test, predictions)
rmse_cv_score = rmse_cv(xgb)

models = pd.concat([models, pd.DataFrame([{
    'Model': 'XGBoost', 'MAE': mae, 'MSE': mse,
    'RMSE': rmse, 'R2 Score': r2, 'RMSE (CV)': rmse_cv_score
}])], ignore_index=True)

print(f'XGBoost - RMSE: {rmse:.2f}, R2: {r2:.4f}')

## Model Comparison

In [ ]:
# Compare models
models_sorted = models.sort_values(by='RMSE (CV)')
models_sorted

In [ ]:
# Visualize model comparison
plt.figure(figsize=(12, 6))
sns.barplot(x='Model', y='RMSE (CV)', data=models_sorted)
plt.title('Model Comparison - RMSE (Cross-Validation)', size=15)
plt.xticks(rotation=30)
plt.ylabel('RMSE')
plt.show()

## Conclusion

We successfully analyzed insurance claims data and built multiple regression models to predict claim amounts. The models were evaluated using various metrics including MAE, MSE, RMSE, and R² Score.

**Key Findings:**
- Identified important factors affecting insurance claims
- Compared multiple regression algorithms
- Evaluated model performance using cross-validation

---
**Author:** [BELYAGOUBIABDELILAH](https://github.com/BELYAGOUBIABDELILAH)